# Глава 16. Проект «Классификатор моих фото»

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь обучить классификатор изображений на собственных данных, применить аугментацию и обернуть модель в веб-интерфейс через Gradio, получив публичную ссылку.

Соберём всё из третьей фазы в настоящий продукт: обучим сеть узнавать категории на твоих собственных фотографиях и завернём её в веб-приложение, ссылку на которое можно отправить родителям. Это первый проект курса, который живёт в интернете и работает на реальных, твоих данных.

## Шаг 1. Собери свои данные

Выбери 2–3 категории, которые тебе интересно различать: например, «кошки / собаки», «кроссовки / ботинки», твои любимые игрушки или марки машин. Сними или скачай по 20–30 фотографий каждой категории и разложи по папкам с названиями категорий. Данные — половина успеха: чем разнообразнее и чётче фото, тем лучше обучится сеть.

**Аналогия: личный датасет  💡**

До сих пор ты работал с чужими, готовыми датасетами. Теперь ты создаёшь свой — с нуля, своими руками, своей камерой. Это ровно то, чем занимаются инженеры машинного обучения в реальных задачах: качество и продуманность данных решают больше, чем выбор модели. Хорошие данные с простой моделью почти всегда бьют плохие данные со сложной.

## Шаг 2. Аугментация данных

**Аналогия: одно фото — много ракурсов  💡**

30 фотографий для сети — маловато, она может просто запомнить их (переобучение). Аугментация (augmentation) искусственно расширяет датасет: слегка поворачивает, отражает, меняет яркость каждой картинки. Из одного фото получается несколько разных вариантов. Сеть видит объект под разными углами и учится узнавать его надёжнее — как если бы ты рассмотрел предмет со всех сторон, а не только строго спереди.

In [ ]:
from tensorflow.keras.layers import RandomFlip, RandomRotation
from tensorflow.keras.models import Sequential

# Слой аугментации: случайный поворот и отражение
augment = Sequential([
    RandomFlip("horizontal"),   # случайное зеркальное отражение
    RandomRotation(0.1),        # случайный поворот на небольшой угол
])

## Шаг 3. Дообучаем MobileNetV2

Берём готовую сеть MobileNetV2 (наш «готовый эксперт» из главы 15), «замораживаем» её накопленные знания и добавляем сверху свой маленький классификатор под наши категории. Это transfer learning в действии — сеть уже умеет видеть формы, нам остаётся научить её различать именно наши классы.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

# Готовый "эксперт", обученный на миллионах картинок ImageNet
base = MobileNetV2(input_shape=(160, 160, 3),
                   include_top=False, weights="imagenet")
base.trainable = False    # ЗАМОРАЖИВАЕМ — не переучиваем с нуля

# Добавляем свою "голову" под наши категории
x = GlobalAveragePooling2D()(base.output)
out = Dense(2, activation="softmax")(x)   # 2 = число твоих категорий
model = Model(base.input, out)

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

**Внимание  ⚠️**

Загрузка готовой модели (weights="imagenet") и работа Gradio требуют доступа в интернет — запускай эти ячейки именно в Google Colab, а не в среде без сети. В Colab всё работает из коробки. Параметр base.trainable = False и есть «заморозка»: без неё мы бы стёрли ценные знания эксперта и потеряли весь смысл transfer learning.

## Шаг 4. Веб-интерфейс через Gradio

Модель обучена — но как её показать другим? Gradio создаёт веб-страницу с загрузкой картинки и ответом модели буквально из нескольких строк Python. И сразу даёт публичную ссылку, которую можно открыть с любого телефона.

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image

labels = ["категория_1", "категория_2"]   # твои названия

def classify(img):
    img = Image.fromarray(img).resize((160, 160))
    arr = np.expand_dims(np.array(img) / 255.0, axis=0)
    pred = model.predict(arr)[0]
    # Возвращаем словарь {метка: вероятность}
    return {labels[i]: float(pred[i]) for i in range(len(labels))}

# Собираем интерфейс: вход — картинка, выход — метки с вероятностями
demo = gr.Interface(fn=classify,
                    inputs=gr.Image(),
                    outputs=gr.Label(),
                    title="Мой классификатор фото")

# share=True даёт публичную ссылку, которой можно поделиться
demo.launch(share=True)

_Ожидаемый результат: ссылка вида https://xxxxx.gradio.live — работающее приложение в браузере._

## Отправь ссылку родителям

**Главная мысль фазы 3  🎯**

Ты прошёл путь от «нейрон — это взвешенная сумма» до собственного веб-приложения, которое узнаёт объекты на фотографиях. Ты понял, как сети видят через матрицы чисел, обучил модель на MNIST, построил CNN и применил transfer learning на своих данных. И, главное, у тебя есть публичная ссылка на живой продукт. Отправь её родителям — это то, что можно показать, чем можно гордиться и что уже похоже на настоящий проект из фазы 6.

## Задания

Базовый уровень

Собери свой датасет из 2 категорий (по 20–30 фото), дообучи MobileNetV2 и запусти Gradio-приложение. Проверь его на новых фото, которых не было в обучении. Отправь ссылку другу.

Со звёздочкой  ⭐

Сделай классификатор из 3 категорий и добавь аугментацию. Сравни точность с аугментацией и без неё на маленьком датасете. Помогла ли аугментация? Затем найди фото, на котором модель ошибается, и подумай почему.

Вопросы для обсуждения  💬

Почему на маленьком датасете аугментация особенно важна?

Что было бы, если бы мы НЕ заморозили базовую модель (base.trainable = True) при 30 фотографиях?

Чем твой классификатор фото уже похож на настоящий продукт, а чего ему не хватает до «взрослого» приложения?

**Проверь себя  ℹ️**

1. Зачем нужна аугментация данных на маленьком датасете?

2. Что значит «заморозить» базовую модель при transfer learning?

3. Что делает параметр share=True в Gradio?

ФАЗА 4

Большие языковые модели

_Понимаем, как работает то, что стоит за ChatGPT, и решаем реальные задачи._

ChatGPT, Claude и другие чат-боты изменили мир за пару лет. В этой фазе ты заглянешь под капот больших языковых моделей (LLM) — без сложной математики поймёшь, как текст превращается в числа и почему модель иногда уверенно врёт. А затем освоишь главный прикладной навык 2026 года: управлять моделью через промпты и решать реальные задачи через API. И это будет не «ещё один чат-бот».